# Capítulo 3 - Exercício 4: Criando um detector de Spam

O objetivo deste notebook é treinar um detector de Spam.

## Plano do exercício

1. Carregar os dados do Spam e do Ham.
2. Separar os dados em treino e teste usando a divisão padrão do dataset.
3. Treinar um classificador binário para detectar o dígito 5.
4. Medir uma acurácia inicial com validação cruzada.
5. Usar `GridSearchCV` para escolher melhores hiperparâmetros.
6. Repetir o processo para o problema multiclasse, classificando todos os dígitos.
7. Avaliar o melhor modelo final no conjunto de teste.

## Configuração

Importamos as bibliotecas usadas no exercício e verificamos versões mínimas. Mantemos uma semente fixa para tornar os resultados mais reprodutíveis.

In [11]:
import sys
import numpy as np

from packaging import version
import sklearn

assert sys.version_info >= (3, 7)
assert version.parse(sklearn.__version__) >= version.parse("1.0.1")

np.random.seed(42)

# Spam e Ham

## Importando os dados

In [12]:
from pathlib import Path

def import_dados(path):
    dados = []
    for email in sorted(Path(path).iterdir()):
        if email.is_file() and email.name[0].isdigit():
            conteudo = email.read_text(encoding="utf-8", errors="replace")
            dados.append(conteudo)
    return dados

easy_ham = import_dados('./emails/easy_ham')
spam_1 = import_dados('./emails/spam_1')


## Dividindo os dados

In [13]:
train_ham, test_ham = easy_ham[:2000], easy_ham[2000:] 
train_spam, test_spam = spam_1[:400], spam_1[400:] 
print(len(train_ham))
print(len(test_ham))
print(len(train_spam))
print(len(test_spam))

2000
551
400
100


## Pipeline modular de pre-processamento

A ideia agora e separar cada tema em uma celula pequena. Isso deixa o codigo mais explicavel e tambem prepara o caminho para usar exatamente o mesmo pre-processamento no treino e no teste.

O fluxo que vamos construir e:

1. montar `X_raw` e `y` com os e-mails brutos;
2. dividir treino/teste com estratificacao;
3. aplicar um extrator de features nos e-mails de treino;
4. aplicar o mesmo extrator nos e-mails de teste;
5. depois conectar esse extrator a um modelo do scikit-learn.


### 1. Imports e padroes compartilhados

Esta celula concentra Bibliotecas e expressoes regulares usadas por varias partes da pipeline

Se voce quiser melhorar a detecao de palavras promocionais, este eh um dos lugares certos para mexer


In [14]:
import re 
from email import policy
from email.parser import Parser
from email.utils import getaddresses
from html.parser import HTMLParser
from urllib.parse import urlparse

import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split

URL_RE = re.compile(r"https?://[^\s<>\"')]+|www\.[^\s<>\"')]+", re.IGNORECASE)

MONEY_RE = re.compile(
    r"(?:R\$|US\$|\$|€|£)\s?\d+(?:[.,]\d+)*|\b\d+(?:[.,]\d+)?\s?(?:%|percent|dollars?|usd|reais)\b",
    re.IGNORECASE,
)

PROMO_RE = re.compile(
     r"\b(?:free|bonus|offer|discount|guarantee|guaranteed|winner|win|cash|money|save|order now|click here|limited|urgent|risk free|no risk)\b",
    re.IGNORECASE,
)

### 2. HTML para TEXTO

Spam costuma usar HTML para layout, cor fontes grandes e botoes

Para o modelo, o mais util normalmente e transformar esse HTML em texto limpo, preservando a mensagem e removendo as tags

In [15]:
class HTMLTextExtractor(HTMLParser):

    def __init__(self):
        super().__init__()
        self.parts = []
        self.skip_depth = 0

    def handle_starttag(self, tag, _attrs):
        tag = tag.lower()
        if tag in {"script","style"}:
            self.skip_depth += 1
        elif tag in {"br","p","div","tr","li"}:
            self.parts.append(" ")
    
    def handle_endtag(self, tag):
        if tag.lower() in {"script", "style"} and self.skip_depth:
            self.skip_depth -= 1

    def handle_data(self, data):
        if not self.skip_depth:
            self.parts.append(data)

    def text(self):
        return re.sub(r"\s+", " ", " ".join(self.parts)).strip()
    
def html_para_texto(html):
    parser = HTMLTextExtractor()
    parser.feed(html or "")
    return parser.text()


### 3. Headers: Remente, Dominio e Assunto

Funcoes para ler os campos: `From`, `Reply-to` e `Subject`

Para reduzir overfitting, a pipeline nao usa o email completo como feature principal, ele extrai sinais mais gerais como o dominio, tamanho da parte inicial(antes do @) e a diferenca entre `From` e `Reply-to`

In [16]:
def parse_email(raw):
    return Parser(policy=policy.defalut).parsestr(raw)


def texto_header(msg, nome):
    return str(msg.get(nome, "") or "")


def primeiro_email(valor_header):
    enderecos = getaddresses([valor_header])
    return enderecos[0][1].lower() if enderecos else ""


def dominio_email(endereco):
    return endereco.rsplit("@", 1)[-1].lower() if "@" in endereco else ""


def parte_local_email(endereco):
    return endereco.split("@", 1)[0].lower() if "@" in endereco else endereco.lower()


def features_remetente(msg):
    from_addr = primeiro_email(texto_header(msg, "From"))
    reply_to_addr = primeiro_email(texto_header(msg, "Reply-to"))
    from_domain = dominio_email(from_addr)
    reply_to_domain = dominio_email(reply_to_addr)
    from_local = parte_local_email(from_addr)

    return{
        "from_addr": from_addr,
        "from_domain": from_domain,
        "reply_to_addr": reply_to_addr,
        "reply-to_domain": reply_to_domain,
        "from_reply_to_domain_diff": bool(reply_to_domain and from_domain and reply_to_domain != from_domain),
        "from_local_len": len(from_local),
        "from_local_digit_count": sum(c.isdigit() for c in from_local),
    }


### 4. Subject

O assunto costuma carregar sinais fortes de spam: urgencia, promocao, dinheiro, excesso de maisculas e `Re:` Falso.

`Re:` falso significa que o assunto parece uma resposta, mas nao ha headers de conversa anterior

In [17]:
def proporcao_maiusculas(texto):
    letras = [c for c in texto or "" if c.isalpha()]
    return sum(c.isupper() for c in letras) / len(letras) if letras else 0


def features_subject(msg):
    subject = texto_header(msg, "Subject")
    subject_starts_re = bool(re.match(r"^\s*re\s*:", subject, re.IGNORECASE))
    has_thread_headrs = bool(texto_header(msg, "In-Reply-To") or texto_header(msg, "References"))

    return{
        "subject": subject,
        "subject_len": len(subject),
        "subject_word_count": len(re.findall(r"\w+", subject)),
        "subject_exclamation_count": subject.count("!"),
        "subject_upper_ratio": proporcao_maiusculas(subject),
        "subject_starts_re": subject_starts_re,
        "subject_re_false": bool(subject_starts_re and not has_thread_headrs)
    }


### 5. MIME, corpo, HTML, imagens e anexos

E-mails podem conter: texto puro, HTML, imagens e anexos.

Esta celula extrai o corpo limpo e sinais estruturais como `has_html`, `has_attachment`, `content_types`, `charsets` e `transfer_encodings`

In [18]:
def decodificar_parte(part):
    try:
        conteudo = part.get_content()
        return conteudo if isinstance(conteudo, str) else ""
    except Exception:
        payload = part.get_payload(decode=True)
        if payload in None:
            return str(part.get_payload() or "")
        charset = part.get_content_charset() or "utf-8"
        try:
            return payload.decode(charset, errors="replace")
        except LookupError:
            return payload.decode("utf-8", errors="replace")
        
def extrair_corpo_e_mime(msg):
    textos = []
    html_textos = []
    html_part_count = 0
    image_part_count = 0
    attachment_count = 0
    content_types = []
    charsets = []
    transfer_encodings = []

    partes = msg.walk() if msg.is_multipart() else [msg]
    for part in partes:
        content_type = part.get_content_type().lower()
        disposition = part.get_content_disposition()
        filename = part.get_filename()
        charset = part.get_content_charset()
        transfer_encoding = str(part.get("Content-Transfer-Encoding", "") or "").lower()

        content_types.append(content_type)
        if charset:
            charsets.append(charset.lower())

        if transfer_encoding:
            transfer_encodings.append(transfer_encoding)

        if disposition == "attachment" or filename:
            attachment_count += 1

        if content_type.startswith("iamgem/"):
            iamge_part_count += 1
            if content_type == "imagem/gif" or (filename or "").lower().endswith(".gif"):
                gif_part_count += 1

        if content_type == "text/plain" and disposition != "attachment":
            textos.append(decodificar_parte(part))
        elif content_type == "text/html" and disposition != "attachment":
            html_part_count += 1
            html_textos.append(html_para_texto(decodificar_parte(part)))

    corpo_limpo = "\n".join(textos + html_textos).strip()
    content_types_unicos = sorted(set(content_type))

    return {
        "corpo_limpo": corpo_limpo,
        "content_types": ",".join(content_types_unicos),
        "charsets": ",".join(sorted(set(charsets))),
        "transfer_encodings": ",".join(sorted(set(transfer_encoding))),
        "has_html": bool(html_part_count or "text/html" in content_types_unicos),
        "html_part_count":html_part_count,
        "has_image_or_gif": image_part_count > 0,
        "image_part_count": image_part_count,
        "gif_part_count": gif_part_count,
        "has_attachment":attachment_count,
        "attachment_count": attachment_count,
    }
            

### 6. Links e dominios nos links

A quantidade de links ajuda, mas o dominio do link costuma ser ainda mais informativo. Ainda assim, dominios especificos podem gerar overfitting. Por isso `link_domains` fica melgher para analise do que como feature direta.

In [19]:
def extrair_dominios_links(texto):
    urls = URL_RE.findall(texto or "")
    dominios = []

    for url in urls:
        url_parseavel = url if url.lower().startswith(("http://", "https://")) else f"http://{url}"
        dominio = urlparse(url_parseavel).netloc.lower()
        if dominio.startswith("www."):
            dominio = dominio[4:]
        if dominio:
            dominios.append(dominio)

    return urls, sorted(set(dominios))

def features_links(raw):
    urls, dominios = extrair_dominios_links(raw)
    return{
        "link_count": len(urls),
        "link_domain_count": len(dominios),
        "link_domains": ",".join(dominios),
    }

### 7. Linguagem promocial, dinheiro e enfase

Medicao de persuasao: palavras promocionais, valores em dinheiro, exclamacoes e maiusculas.

In [20]:
def features_linguagem(texto_modelo, corpo_limpo):
    return{
        "promo_word_count": len(PROMO_RE.findall(texto_modelo)),
        "money_count": len(MONEY_RE.findall(texto_modelo)),
        "body_exclamation_count": corpo_limpo.count("!"),
        "body_upper_ratio": proporcao_maiusculas(corpo_limpo),
        "raw_text_len": len(texto_modelo)
    }

### 8. Juntando as features de um e-mail

Recebe um e-mail bruto e devolve um dicionario de features

In [21]:
def extrari_features_email(raw):
    msg = parse_email(raw)

    remetente = features_remetente(msg)
    subject_info = features_subject(msg)
    mime = extrair_corpo_e_mime(msg)
    links = features_links(raw)

    texto_modelo = f"{subject_info['subject']}\n{mime['corpo_limpo']}".strip()
    linguagem = features_linguagem(texto_modelo, mime["corpo_limpo"])

    features = {
        "texto_modelo": texto_modelo,
        "has_x_spam_header": bool(re.search(r"^X-Spam-", raw, re.IGNORECASE | re.MULTILINE)),
        "raw_len": len(raw),
    }

    features.update(remetente)
    features.update(subject_info)
    features.update(mime)
    features.update(links)
    features.update(linguagem)
    
    return features
